# Step 7 - Policy Report Charts Walkthrough

This notebook replaces the old `src/generate_policy_report_charts.py` script.

What this notebook does:
- loads the processed annual and country datasets
- defines the five minister-facing matplotlib charts
- saves the chart assets into `outputs/charts/`


## Outline

1. Set up project paths and plotting defaults
2. Load the annual and country datasets
3. Build the five chart helper functions
4. Run the helpers and confirm the output files


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    path
    for path in candidates
    if (path / 'data').exists() and (path / 'outputs').exists()
)

ANNUAL_FILE = PROJECT_ROOT / 'data' / 'processed' / 'india_semiconductor_integrated_annual.csv'
COUNTRY_FILE = PROJECT_ROOT / 'data' / 'processed' / 'india_semiconductor_country_year_breakdown.csv'
RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'india_imports_8542_3818_filtered.csv'
CHART_DIR = PROJECT_ROOT / 'outputs' / 'charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        'figure.dpi': 150,
        'savefig.dpi': 150,
        'savefig.bbox': 'tight',
        'axes.spines.top': False,
        'axes.spines.right': False,
    }
)


## Load the report inputs

The annual table gives us the aggregate import series and product mix. The country table powers the exporter ranking and supplier-share trend charts. We also load the raw BACI extract once so the notebook fails early if that file is missing from the repo.


In [2]:
annual = pd.read_csv(ANNUAL_FILE).sort_values('year').reset_index(drop=True)
country = pd.read_csv(COUNTRY_FILE).sort_values(['year', 'import_value_usd'], ascending=[True, False])
raw = pd.read_csv(RAW_FILE)
latest_year = int(annual['year'].max())

annual[['year', 'real_value_2015usd_billions', 'hs8542_import_usd', 'hs3818_import_usd']].tail()


,year,real_value_2015usd_billions,hs8542_import_usd,hs3818_import_usd
25,2020,7.0684,8.421261e+09,89790460.0
26,2021,9.8664,1.244516e+10,140386684.0
27,2022,11.8095,1.633338e+10,181048513.0
28,2023,12.7337,1.867590e+10,331631060.0
29,2024,14.5977,2.286137e+10,188435605.0


## Chart helpers for total imports and product mix

These two charts answer the first executive questions: how large the import bill has become in real terms, and how that bill splits across finished chips versus semiconductor materials.


In [3]:
def save_total_imports_chart(annual: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(annual['year'], annual['real_value_2015usd_billions'], linewidth=2)
    ax.set_title("Total Semiconductor Imports into India, 1995-2024")
    ax.set_xlabel('Year')
    ax.set_ylabel('Real imports (2015 USD billions)')
    ax.grid(True, alpha=0.3)
    fig.savefig(CHART_DIR / 'minister_chart_1_total_imports_real.png')
    plt.close(fig)


def save_product_mix_chart(annual: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 4.8))
    hs8542_bn = annual['hs8542_import_usd'] / 1e9
    hs3818_bn = annual['hs3818_import_usd'] / 1e9
    ax.bar(annual['year'], hs8542_bn, label='HS 8542: Integrated circuits')
    ax.bar(annual['year'], hs3818_bn, bottom=hs8542_bn, label='HS 3818: Semiconductor materials')
    ax.set_title("India's Imports by Product Type, 1995-2024")
    ax.set_xlabel('Year')
    ax.set_ylabel('Nominal imports (USD billions)')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    fig.savefig(CHART_DIR / 'minister_chart_2_product_mix.png')
    plt.close(fig)


## Chart helpers for exporter ranking and supplier trends

The next pair shifts from total scale to supply structure. One chart shows who the biggest exporters are in the latest year, and the other shows how the main supplier shares evolve over time.


In [4]:
def save_top_exporters_chart(country: pd.DataFrame, latest_year: int) -> None:
    latest = (
        country[(country['year'] == latest_year) & (country['exporter_name'] != 'Other')]
        .sort_values('import_value_usd', ascending=False)
        .head(10)
        .sort_values('import_value_usd')
        .copy()
    )
    latest['import_value_bn'] = latest['import_value_usd'] / 1e9

    fig, ax = plt.subplots(figsize=(10, 5.2))
    bars = ax.barh(latest['exporter_name'], latest['import_value_bn'])
    ax.set_title(f'Top Exporting Countries to India in {latest_year}')
    ax.set_xlabel('Nominal imports (USD billions)')
    ax.set_ylabel('Exporter')
    ax.grid(True, axis='x', alpha=0.3)
    for bar, share in zip(bars, latest['market_share_pct']):
        ax.text(bar.get_width() + 0.08, bar.get_y() + bar.get_height() / 2, f'{share:.1f}%', va='center', fontsize=8)
    fig.savefig(CHART_DIR / 'minister_chart_3_top_exporters_2024.png')
    plt.close(fig)


def save_market_share_trends_chart(country: pd.DataFrame) -> None:
    selected = ['China', 'Taiwan', 'South Korea', 'Singapore', 'Hong Kong']
    pivot = (
        country[country['exporter_name'].isin(selected)]
        .pivot_table(index='year', columns='exporter_name', values='market_share_pct', aggfunc='sum')
        .fillna(0)
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(10, 4.8))
    for name in selected:
        ax.plot(pivot.index, pivot[name], linewidth=2, label=name)
    ax.set_title('Market Share Trends of Major Suppliers, 1995-2024')
    ax.set_xlabel('Year')
    ax.set_ylabel('Market share (%)')
    ax.legend(ncol=3, fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.savefig(CHART_DIR / 'minister_chart_4_market_share_trends.png')
    plt.close(fig)


## Final helper, execution, and output check

The last chart summarizes growth dynamics. After defining it, we run all five helpers and list the generated files so the notebook ends with a quick verification step.


In [5]:
def save_growth_chart(annual: pd.DataFrame) -> None:
    growth = annual[['year', 'yoy_real_growth_pct']].dropna().copy()
    rolling = growth['yoy_real_growth_pct'].rolling(3, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.bar(growth['year'], growth['yoy_real_growth_pct'], label='Annual real growth')
    ax.plot(growth['year'], rolling, linewidth=2, label='3-year rolling average')
    ax.axhline(0, linewidth=1)
    ax.set_title("Growth Rate of India's Semiconductor Imports, 1996-2024")
    ax.set_xlabel('Year')
    ax.set_ylabel('YoY real growth (%)')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    fig.savefig(CHART_DIR / 'minister_chart_5_growth_rate.png')
    plt.close(fig)


save_total_imports_chart(annual)
save_product_mix_chart(annual)
save_top_exporters_chart(country, latest_year)
save_market_share_trends_chart(country)
save_growth_chart(annual)

sorted(path.name for path in CHART_DIR.glob('minister_chart_*.png'))


['minister_chart_1_total_imports_real.png',
 'minister_chart_2_product_mix.png',
 'minister_chart_3_top_exporters_2024.png',
 'minister_chart_4_market_share_trends.png',
 'minister_chart_5_growth_rate.png']